# Phase 1.3: Synthetic Data Generation - CO2/PM10/COV

## Objectif
Générer données synthétiques calibrées pour CO2, PM10, COV (manquants dans sources publiques).

## Approche
- CO2: Corrélation avec NO2 (r ≈ 0.75)
- PM10: Basé sur PM2.5 avec ajustement approprié (ratio 1.2x)
- COV: Corrélation légère avec polluants gaz

## Sortie
- `ia/data/training_dataset.csv`: Dataset complet avec synthétiques marqués

## Section 1: Setup

In [ ]:
# Imports et configuration de base
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import io
import zipfile
import requests
from pathlib import Path

# Réproducibilité
np.random.seed(42)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Option: prioriser ingestion EPA AQS pour PM10 / VOCs (COV).
# Si False, on utilisera directement la génération synthétique pour PM10/COV.
use_epa = False

# Fonctions utilitaires pour lister et (éventuellement) télécharger fichiers EPA
def list_epa_files(file_list_url='https://aqs.epa.gov/aqsweb/airdata/file_list.csv'):
    try:
        r = requests.get(file_list_url, timeout=30)
        r.raise_for_status()
        df_files = pd.read_csv(io.StringIO(r.text))
        return df_files
    except Exception as e:
        print('Impossible de récupérer file_list.csv depuis EPA:', e)
        return pd.DataFrame()

def download_and_extract_csv(file_name, base_url='https://aqs.epa.gov/aqsweb/airdata/'):
    url = base_url + file_name
    print('Téléchargement EPA:', url)
    r = requests.get(url, timeout=120)
    r.raise_for_status()
    z = zipfile.ZipFile(io.BytesIO(r.content))
    csv_names = [n for n in z.namelist() if n.lower().endswith('.csv')]
    if not csv_names:
        raise RuntimeError('Aucun CSV trouvé dans le ZIP EPA')
    dfs = []
    for name in csv_names:
        with z.open(name) as f:
            dfs.append(pd.read_csv(f, low_memory=False))
    return pd.concat(dfs, ignore_index=True)

def _epa_col(df, *candidates):
    """Retourne la première colonne présente (insensible à la casse)."""
    lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in lower:
            return lower[name.lower()]
    return None


def get_epa_filename_series(df_files):
    col = _epa_col(df_files, 'Filename', 'file_name', 'filename')
    if col is None:
        raise KeyError(
            f"Colonne nom de fichier absente dans file_list.csv. Colonnes: {list(df_files.columns)}"
        )
    return df_files[col].astype(str)


def map_epa_to_canonical(df):
    # Mappage minimal vers le schéma canonique (colonnes EPA hourly standard)
    df2 = df.copy()
    df2.columns = [str(c).strip() for c in df2.columns]
    col_map = {c.lower(): c for c in df2.columns}

    def pick(*names):
        for n in names:
            if n.lower() in col_map:
                return col_map[n.lower()]
        return None

    val_col = pick('Sample Measurement', 'sample_measurement')
    if val_col:
        df2['value'] = pd.to_numeric(df2[val_col], errors='coerce')

    date_col = pick('Date Local', 'date_local')
    time_col = pick('Time Local', 'time_local')
    if date_col and time_col:
        df2['timestamp_utc'] = pd.to_datetime(
            df2[date_col].astype(str) + ' ' + df2[time_col].astype(str), errors='coerce'
        )
    elif date_col:
        df2['timestamp_utc'] = pd.to_datetime(df2[date_col], errors='coerce')

    param_col = pick('Parameter Name', 'parameter_name')
    if param_col:
        df2['pollutant'] = df2[param_col].astype(str).str.upper()
        df2.loc[df2['pollutant'].str.contains('PM10', na=False), 'pollutant'] = 'PM10'
        df2.loc[df2['pollutant'].str.contains('VOC|NMOC|COV', na=False), 'pollutant'] = 'COV'

    unit_col = pick('Units of Measure', 'units_of_measure')
    if unit_col:
        df2['unit'] = df2[unit_col]

    site_col = pick('Site Num', 'site_id', 'site_number', 'monitor_id')
    if site_col:
        df2['site_id'] = df2[site_col].astype(str)

    keep = [c for c in ['timestamp_utc', 'site_id', 'pollutant', 'value', 'unit'] if c in df2.columns]
    return df2[keep].dropna(subset=['timestamp_utc', 'value'], how='any')


def select_epa_hourly_files(fnames, pattern, prefer_code=None, max_files=2):
    """Sélectionne les N fichiers hourly les plus récents correspondant au motif."""
    s = pd.Series(fnames.astype(str))
    mask = s.str.contains(pattern, case=False, na=False)
    if prefer_code:
        prefer = s.str.contains(prefer_code, case=False, na=False)
        mask = mask & prefer
    matched = s[mask].sort_values(ascending=False)
    return matched.head(max_files).tolist()


# Essai de lister les fichiers EPA (ne télécharge pas encore les gros zips)
epa_file_list = list_epa_files() if use_epa else pd.DataFrame()
pm10_candidates = []
voc_candidates = []
if not epa_file_list.empty:
    fnames = get_epa_filename_series(epa_file_list)
    pm10_candidates = select_epa_hourly_files(
        fnames, r'hourly_81102|hourly_PM10', prefer_code='81102', max_files=2
    )
    voc_candidates = select_epa_hourly_files(
        fnames, r'hourly_VOCS|hourly_43102|hourly_43502', max_files=2
    )
    print(
        f'EPA file_list trouvé: {len(fnames)} entrées — '
        f'pm10_candidates={len(pm10_candidates)}, voc_candidates={len(voc_candidates)}'
    )

print('✅ Imports et configuration initiale prêts — use_epa=', use_epa)

KeyError: 'file_name'

In [ ]:
# Configuration des chemins
PROJECT_ROOT = Path("../").resolve()
DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"

cleaned_file = DATA_DIR / "cleaned_features.csv"
output_file = DATA_DIR / "training_dataset.csv"

print(f"📂 Input: {cleaned_file}")
print(f"📂 Output: {output_file}")

In [ ]:
# Charger données nettoyées
print("🔄 Chargement données nettoyées...\n")

df = pd.read_csv(cleaned_file)

print(f"✅ Données chargées: {len(df)} lignes")
print(f"   Colonnes: {list(df.columns)[:10]}...")
print(f"   Polluants présents: {sorted(df['pollutant'].unique())}")

In [ ]:
# Si use_epa=True et qu'il y a des candidats, tenter de télécharger et mapper PM10/VOC
epa_pm10_df = None
epa_voc_df = None
if use_epa and len(pm10_candidates) > 0:
    try:
        fn = pm10_candidates[0]
        raw = download_and_extract_csv(fn)
        epa_pm10_df = map_epa_to_canonical(raw)
        print('EPA PM10 téléchargé et mappé, rows=', len(epa_pm10_df))
    except Exception as e:
        print('Échec ingestion PM10 EPA:', e)
        epa_pm10_df = None
if use_epa and len(voc_candidates) > 0:
    try:
        fn = voc_candidates[0]
        raw = download_and_extract_csv(fn)
        epa_voc_df = map_epa_to_canonical(raw)
        print('EPA VOCs téléchargé et mappé, rows=', len(epa_voc_df))
    except Exception as e:
        print('Échec ingestion VOCs EPA:', e)
        epa_voc_df = None

# Normalisation: forcer pollutant name conventions si présent
if epa_pm10_df is not None and 'pollutant' in epa_pm10_df.columns:
    epa_pm10_df['pollutant'] = epa_pm10_df['pollutant'].str.replace('PM10', 'PM10', case=False)
if epa_voc_df is not None and 'pollutant' in epa_voc_df.columns:
    epa_voc_df['pollutant'] = epa_voc_df['pollutant'].str.replace('VOC', 'COV', case=False)

print('✅ Tentative dingestion EPA terminée (si disponible)')

## Section 2: Génération CO2 Synthétique

In [ ]:
# ========================================
# Section 2: Génération CO2 Synthétique
# ========================================
# CO2 n'est pas mesuré dans les sources publiques EPA/Beijing/UCI
# On le génère synthétiquement en utilisant une corrélation avec NOX (mg/Nm3)

print("🔄 Génération CO2 synthétique...\n")

# Créer un tableau pivot: polluants en colonnes, timestamps en lignes
df_pivot = df.pivot_table(
    index='timestamp_utc',
    columns='pollutant',
    values='value_smooth' if 'value_smooth' in df.columns else 'value',
    aggfunc='mean',
)

# CO2 (ppm) ~ baseline + faible terme NOX (mg/Nm3) + bruit
nox_proxy = df_pivot.get('NOX', df_pivot.get('NO2'))
if nox_proxy is not None:
    co2_baseline = 420
    co2_slope = 0.05          # ppm par mg/Nm3 NOX (ordre de grandeur industriel)
    co2_noise_std = 20
    default_nox = float(nox_proxy.median()) if nox_proxy.notna().any() else 100.0

    df_pivot['CO2'] = (
        co2_baseline
        + co2_slope * nox_proxy.fillna(default_nox)
        + np.random.normal(0, co2_noise_std, len(df_pivot))
    )
    df_pivot['CO2'] = df_pivot['CO2'].clip(300, 5000)

    print("✅ CO2 synthétique généré")
    print(f"   Mean: {df_pivot['CO2'].mean():.1f} ppm")
    print(f"   Std: {df_pivot['CO2'].std():.1f} ppm")
    print(f"   Range: [{df_pivot['CO2'].min():.1f}, {df_pivot['CO2'].max():.1f}] ppm")
else:
    print("⚠️  NOX non trouvé, CO2 non généré")

## Section 3: Génération PM10 Synthétique

In [ ]:
# PM10: prioriser EPA si présent, sinon générer synthétique basé sur PM2.5
print("🔄 Traitement PM10 (EPA-first puis fallback synthétique)...\n")
# Si EPA a fourni des valeurs PM10 mappées, les utiliser en priorité
try:
    if 'epa_pm10_df' in globals() and epa_pm10_df is not None and not epa_pm10_df.empty:
        print('→ Utilisation des données PM10 provenant dEPA (prioritaire)')
        series = epa_pm10_df.copy()
        series = series.set_index('timestamp_utc')['value']
        # aligner avec l'index temporel de df_pivot (index = timestamp_utc)
        df_pivot['PM10'] = df_pivot.index.to_series().map(series).astype(float)
        # si des trous subsistent, on pourra compléter avec synthétique ci-dessous
    else:
        raise KeyError('epa_pm10_df absent')
except Exception:
    # Fallback synthétique à partir de PM2.5 si EPA non disponible
    print('→ EPA PM10 non utilisable; génération synthétique à partir de PM2.5')
    if 'PM25' in df_pivot.columns:
        pm10_ratio = 1.2
        pm10_noise_std = 5
        synth = pm10_ratio * df_pivot['PM25'].fillna(0) + np.random.normal(0, pm10_noise_std, len(df_pivot))
        df_pivot['PM10'] = df_pivot.get('PM10', np.nan)
        # remplir les valeurs manquantes avec synth
        df_pivot['PM10'] = df_pivot['PM10'].fillna(pd.Series(synth, index=df_pivot.index))
        df_pivot['PM10'] = df_pivot['PM10'].clip(0, 50)
        print(f"✅ PM10 synthétique (fallback) généré / complété")
    else:
        print('⚠️ PM2.5 non trouvé — impossible de générer PM10 synthétique')

## Section 4: Génération COV Synthétique

In [ ]:
# SECTION: PM10 processing (EPA-first with fallback)
print("🔄 PM10: vérification EPA puis fallback synthétique...\n")
if 'epa_pm10_df' in globals() and epa_pm10_df is not None and not epa_pm10_df.empty:
    print('→ Remplissage PM10 depuis EPA')
    series = epa_pm10_df.set_index('timestamp_utc')['value']
    df_pivot['PM10'] = df_pivot.index.to_series().map(series).astype(float)
    # Compléter trous éventuels par méthode synthétique basée sur PM2.5
    if 'PM25' in df_pivot.columns:
        missing = df_pivot['PM10'].isna()
        if missing.any():
            pm10_ratio = 1.2
            pm10_noise_std = 5
            synth = pm10_ratio * df_pivot['PM25'].fillna(0) + np.random.normal(0, pm10_noise_std, len(df_pivot))
            df_pivot.loc[missing, 'PM10'] = pd.Series(synth, index=df_pivot.index)[missing]
            df_pivot['PM10'] = df_pivot['PM10'].clip(0, 50)
else:
    # EPA non disponible: génération synthétique complète si PM2.5 existe
    print('→ EPA non disponible: génération PM10 synthétique à partir de PM2.5')
    if 'PM25' in df_pivot.columns:
        pm10_ratio = 1.2
        pm10_noise_std = 5
        df_pivot['PM10'] = pm10_ratio * df_pivot['PM25'].fillna(0) + np.random.normal(0, pm10_noise_std, len(df_pivot))
        df_pivot['PM10'] = df_pivot['PM10'].clip(0, 50)
    else:
        print('⚠️ PM2.5 absent — PM10 non disponible')

In [ ]:
# COV: prioriser EPA si présent, sinon génération synthétique basée sur NOX (mg/Nm3)
print("🔄 Traitement COV (EPA-first puis fallback)...\n")
nox_proxy = df_pivot.get('NOX', df_pivot.get('NO2'))

def _synth_cov(nox_series):
    return 20 + 0.15 * nox_series.fillna(0) + np.random.normal(0, 10, len(nox_series))

if 'epa_voc_df' in globals() and epa_voc_df is not None and not epa_voc_df.empty:
    print('→ Utilisation des données VOCs EPA (converties en COV)')
    series = epa_voc_df.set_index('timestamp_utc')['value']
    if 'COV' not in df_pivot.columns:
        df_pivot['COV'] = np.nan
    mapped = df_pivot.index.to_series().map(series).astype(float)
    df_pivot['COV'] = df_pivot['COV'].combine_first(mapped)
    if nox_proxy is not None:
        missing = df_pivot['COV'].isna()
        if missing.any():
            synth = _synth_cov(nox_proxy)
            df_pivot.loc[missing, 'COV'] = pd.Series(synth, index=df_pivot.index)[missing]
elif nox_proxy is not None and 'COV' not in df_pivot.columns:
    print('→ EPA VOC non disponible: génération COV synthétique')
    df_pivot['COV'] = _synth_cov(nox_proxy).clip(0, 200)
elif 'COV' in df_pivot.columns:
    print('→ COV déjà présent dans cleaned_features; pas de génération supplémentaire')
else:
    print('⚠️ NOX absent — impossible de générer COV synthétique')

## Section 5: Validation Synthétiques

In [ ]:
# Validation qualité synthétiques
print("🔍 Validation qualité synthétiques...\n")

for pollutant in ['CO2', 'PM10', 'COV']:
    if pollutant in df_pivot.columns:
        data = df_pivot[pollutant].dropna()
        print(f"\n{pollutant}:")
        print(f"  Count: {len(data)} (non-null)")
        print(f"  Skewness: {data.skew():.2f}")
        print(f"  Kurtosis: {data.kurtosis():.2f}")
        print(f"  CV (Coeff. Variation): {(data.std() / data.mean()):.2f}")

In [ ]:
# Visualiser distributions synthétiques
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

synthetic_pollutants = ['CO2', 'PM10', 'COV']
for idx, pollutant in enumerate(synthetic_pollutants):
    if pollutant in df_pivot.columns:
        data = df_pivot[pollutant].dropna()
        axes[idx].hist(data, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
        axes[idx].set_title(f'{pollutant} Distribution', fontweight='bold')
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(MODELS_DIR / 'synthetic_distributions.png', dpi=100)
plt.show()

print("✅ Distributions visualisées")

## Section 6: Reformatage et Export

In [ ]:
# Reformer en format long (pollutant par ligne)
print("🔄 Reformatage données...\n")

df_long = df_pivot.reset_index().melt(
    id_vars=['timestamp_utc'],
    var_name='pollutant',
    value_name='value'
)

# Marquer synthétiques
df_long['synthetic'] = df_long['pollutant'].isin(['CO2', 'PM10', 'COV'])

# Fusionner avec métadonnées disponibles (colonnes présentes uniquement)
meta_cols = [
    'timestamp_utc', 'site_id', 'site_name', 'temperature_c', 'humidity_percent',
    'pressure_hpa', 'wind_speed_ms', 'source_name', 'value_smooth', 'value_mean_6h',
    'value_std_6h', 'value_max_12h', 'value_roc_1h', 'hour_sin', 'hour_cos',
    'dow_sin', 'dow_cos', 'value_normalized',
]
meta_cols = [c for c in meta_cols if c in df.columns]
df_meta = df[meta_cols].drop_duplicates(subset=['timestamp_utc'], keep='first')
df_long = df_long.merge(df_meta, on='timestamp_utc', how='left')

print(f"✅ Données reformatées")
print(f"   Total lignes: {len(df_long)}")
print(f"   Polluants: {sorted(df_long['pollutant'].unique())}")
print(f"   Synthétiques: {df_long['synthetic'].sum()} ({df_long['synthetic'].sum()/len(df_long)*100:.1f}%)")

In [ ]:
# Exporter fichier final
print(f"💾 Export final...\n")

df_long.to_csv(output_file, index=False)

print(f"✅ Fichier exporté: {output_file}")
print(f"   Taille: {output_file.stat().st_size / 1024 / 1024:.2f} MB")
print(f"   Shape: {df_long.shape}")

## ✅ Résumé - Synthetic Data Generation

✅ CO2 synthétique généré (corrélation NO2)
✅ PM10 synthétique généré (ratio 1.2x PM2.5)
✅ COV synthétique généré (corrélation NO2)
✅ Données marquées avec flag `synthetic`
✅ Dataset complet exporté → `ia/data/training_dataset.csv`

## 🎯 Prochaines Étapes
1. Notebook 04: Entraîner Isolation Forest
2. Notebook 05: Préparer tenseurs LSTM
3. Notebooks 06-07: Entraîner LSTM 1h et 24h